# Hospital Bed Allocation and Patient Flow Simulation
### Base Model — Discrete-Event Simulation

This notebook implements the **baseline** hospital simulation (10 beds, queue capacity 5,
FIFO discipline, uniform(1,3)h arrivals) as a custom discrete-event engine, following the
time/counter/state-variable + event-list formalism from the course material.

Scenario experiments (capacity, queue policy, demand change) are intentionally **not**
included here yet — this notebook covers only what's required for the base case:
simulation logic, verification checks, the two required output tables, Section 4 metrics
with 95% confidence intervals over 30+ replications, and the base-run visualizations.

## 1. Imports & Configuration

In [ ]:
import heapq
import itertools
from dataclasses import dataclass, asdict
from typing import Optional

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

%matplotlib inline


In [ ]:
# ----------------------------- Baseline configuration -----------------------------
HORIZON = 24 * 30                     # 30-day horizon, in hours
NUM_BEDS = 10
QUEUE_CAPACITY = 5
ARRIVAL_LOW, ARRIVAL_HIGH = 1.0, 3.0  # uniform interarrival time (hours)
TRIAGE_LOW, TRIAGE_HIGH = 0.1, 0.3    # uniform triage time (hours), i.e. 6-18 minutes

N_REPLICATIONS = 30
BASE_SEED = 2026


## 2. Appendix A Data Tables

Disease and subtype probabilities are equal, per the assignment's stated default
(no strong clinical justification was available to deviate from this).

In [ ]:
# Age groups: (label, probability, age_priority)
AGE_GROUPS = [
    ("0-18",  0.15, 3),
    ("19-35", 0.25, 2),
    ("36-55", 0.35, 1),
    ("56-75", 0.15, 4),
    ("76+",   0.10, 5),
]
AGE_LABELS = [a[0] for a in AGE_GROUPS]
AGE_PROBS = [a[1] for a in AGE_GROUPS]
AGE_PRIORITY = {a[0]: a[2] for a in AGE_GROUPS}

# Diseases: base_severity, treatment-time range (hours), subtypes ordered mild -> severe
DISEASES = {
    "Broken Arm":      {"base_severity": 2, "range": (12, 24),
                         "subtypes": ["Hairline Fracture", "Simple Fracture", "Compound Fracture"]},
    "Concussion":      {"base_severity": 4, "range": (18, 24),
                         "subtypes": ["Mild", "Moderate", "Severe"]},
    "Simple Fracture": {"base_severity": 3, "range": (12, 18),
                         "subtypes": ["Non-displaced", "Displaced", "Comminuted"]},
    "Appendicitis":    {"base_severity": 5, "range": (12, 24),
                         "subtypes": ["Early Stage", "Acute", "Perforated"]},
    "Pneumonia":       {"base_severity": 4, "range": (18, 24),
                         "subtypes": ["Mild", "Moderate", "Severe"]},
}
DISEASE_LABELS = list(DISEASES.keys())
SUBTYPE_MODIFIER = {0: -1, 1: 0, 2: 1}  # 1st subtype = mild (-1), 2nd = mid (0), 3rd = severe (+1)

# "Normal" vital sign ranges used to flag abnormal readings for the severity score
BP_NORMAL = (90, 140)
PULSE_NORMAL = (60, 100)
TEMP_NORMAL = (36.1, 37.8)


## 3. Patient Record

One `Patient` object per arrival; its fields map directly onto the required
patient-level output table (Section 4).

In [ ]:
@dataclass
class Patient:
    patient_id: int
    arrival_time: float
    age_group: str
    age_priority: int
    disease: str
    subtype: str
    bp: float
    pulse: float
    temp: float
    severity_score: float
    priority_score: float
    priority_level: str
    triage_time: float
    treatment_duration: Optional[float] = None
    queue_entry_time: Optional[float] = None
    wait_to_bed: Optional[float] = None
    bed_number: Optional[int] = None
    admission_time: Optional[float] = None
    discharge_time: Optional[float] = None
    total_time_in_system: Optional[float] = None
    status: Optional[str] = None            # 'admitted' | 'rejected'
    rejection_reason: Optional[str] = None
    in_queue: bool = False                  # internal bookkeeping only


## 4. Attribute Sampling & Scoring Formulas

Distributional choices and the severity / priority / safe-waiting-time formulas are
documented and justified in the accompanying `Design_Decisions.md`. Summary:

- **Vitals**: Normal(120,15)/Normal(80,15)/Normal(37.0,0.6) for BP/pulse/temp, clipped to
  plausible ranges.
- **Triage time**: Uniform(0.1, 0.3) hours.
- **Severity score** = disease base severity + subtype modifier (-1/0/+1) + count of
  abnormal vitals (0-3) + age factor ((age_priority-1)*0.5).
- **Priority score** = 0.7·severity + 0.3·(age_priority·2); mapped to low/medium/high.
- **Safe waiting time** = max(1, 12 − priority_score) hours — higher priority → shorter wait.
- **Treatment duration** = Triangular(low, mode, high) from the disease's range, scaled by
  a severity multiplier (0.8 + 0.04·severity_score).

In [ ]:
def sample_age_group(rng):
    label = rng.choice(AGE_LABELS, p=AGE_PROBS)
    return label, AGE_PRIORITY[label]


def sample_disease_and_subtype(rng):
    disease = rng.choice(DISEASE_LABELS)             # equal probability across diseases
    subtypes = DISEASES[disease]["subtypes"]
    idx = rng.integers(0, len(subtypes))              # equal probability within disease
    return disease, subtypes[idx], SUBTYPE_MODIFIER[idx]


def sample_vitals(rng):
    bp = float(np.clip(rng.normal(120, 15), 70, 200))
    pulse = float(np.clip(rng.normal(80, 15), 40, 180))
    temp = float(np.clip(rng.normal(37.0, 0.6), 35, 41))
    return bp, pulse, temp


def count_abnormal_vitals(bp, pulse, temp):
    count = 0
    if not (BP_NORMAL[0] <= bp <= BP_NORMAL[1]):
        count += 1
    if not (PULSE_NORMAL[0] <= pulse <= PULSE_NORMAL[1]):
        count += 1
    if not (TEMP_NORMAL[0] <= temp <= TEMP_NORMAL[1]):
        count += 1
    return count


def compute_severity(disease, subtype_modifier, vital_abnormal_count, age_priority):
    base = DISEASES[disease]["base_severity"]
    age_factor = (age_priority - 1) * 0.5
    return base + subtype_modifier + vital_abnormal_count + age_factor


def compute_priority(severity_score, age_priority):
    score = 0.7 * severity_score + 0.3 * (age_priority * 2)
    if score >= 8:
        level = "high"
    elif score >= 4:
        level = "medium"
    else:
        level = "low"
    return score, level


def safe_waiting_time(priority_score):
    return max(1.0, 12.0 - priority_score)


def sample_treatment_duration(rng, disease, severity_score):
    low, high = DISEASES[disease]["range"]
    mode = (low + high) / 2
    base = rng.triangular(low, mode, high)
    multiplier = 0.8 + 0.04 * severity_score
    return base * multiplier


def generate_patient(rng, patient_id, arrival_time):
    age_group, age_priority = sample_age_group(rng)
    disease, subtype, subtype_mod = sample_disease_and_subtype(rng)
    bp, pulse, temp = sample_vitals(rng)
    vital_abnormal = count_abnormal_vitals(bp, pulse, temp)
    severity = compute_severity(disease, subtype_mod, vital_abnormal, age_priority)
    priority_score, priority_level = compute_priority(severity, age_priority)
    triage_time = rng.uniform(TRIAGE_LOW, TRIAGE_HIGH)
    return Patient(
        patient_id=patient_id, arrival_time=arrival_time,
        age_group=age_group, age_priority=age_priority,
        disease=disease, subtype=subtype,
        bp=bp, pulse=pulse, temp=temp,
        severity_score=severity, priority_score=priority_score,
        priority_level=priority_level, triage_time=triage_time,
    )


## 5. Discrete-Event Simulation Engine

Follows the Section 7.1 pattern directly:

- **Time variable**: `t`, advanced to the next event's time (next-event time advance).
- **System state (`Hospital`)**: beds used, free bed-number pool, FIFO queue, patient records.
- **Event list (FEL)**: a `heapq`-based priority queue of `(time, seq, event_type, patient_id)`.
- **Events**: `ARRIVAL`, `TRIAGE_DONE`, `QUEUE_TIMEOUT`, `DISCHARGE`, `MONITOR` (hourly snapshot).
  `ADMIT`/`REJECT` are synchronous state transitions inside the other handlers, not
  separately scheduled events.

**End-of-horizon convention (Section 3.5)**: arrivals stop being generated at t=720h, but
patients already in the system (admitted or queued) are allowed to run to natural
completion — the *drain* approach. No censoring is needed under this convention.

In [ ]:
class Hospital:
    """Holds the system state SS."""
    def __init__(self, num_beds, queue_capacity):
        self.num_beds = num_beds
        self.queue_capacity = queue_capacity
        self.beds_used = 0
        self.free_bed_numbers = list(range(1, num_beds + 1))
        self.queue = []          # FIFO list of patient_ids currently queued
        self.patients = {}       # patient_id -> Patient
        self.hourly_records = []


def run_single_replication(seed, horizon=HORIZON, num_beds=NUM_BEDS,
                            queue_capacity=QUEUE_CAPACITY,
                            arrival_low=ARRIVAL_LOW, arrival_high=ARRIVAL_HIGH):
    rng = np.random.default_rng(seed)
    hospital = Hospital(num_beds, queue_capacity)
    id_gen = itertools.count(1)

    fel = []
    seq_counter = itertools.count()

    def schedule(time, event_type, patient_id=None):
        heapq.heappush(fel, (time, next(seq_counter), event_type, patient_id))

    def admit(patient, t):
        patient.wait_to_bed = t - patient.queue_entry_time
        patient.bed_number = hospital.free_bed_numbers.pop(0)
        patient.admission_time = t
        patient.status = "admitted"
        patient.in_queue = False
        hospital.beds_used += 1
        patient.treatment_duration = sample_treatment_duration(rng, patient.disease, patient.severity_score)
        schedule(t + patient.treatment_duration, "DISCHARGE", patient.patient_id)

    def reject(patient, reason, t):
        patient.status = "rejected"
        patient.rejection_reason = reason
        patient.in_queue = False
        patient.total_time_in_system = t - patient.arrival_time

    def handle_arrival(t):
        pid = next(id_gen)
        patient = generate_patient(rng, pid, t)
        hospital.patients[pid] = patient
        schedule(t + patient.triage_time, "TRIAGE_DONE", pid)
        gap = rng.uniform(arrival_low, arrival_high)
        if t + gap < horizon:
            schedule(t + gap, "ARRIVAL")

    def handle_triage_done(t, pid):
        patient = hospital.patients[pid]
        patient.queue_entry_time = t
        if hospital.beds_used < hospital.num_beds:
            admit(patient, t)
        elif len(hospital.queue) < hospital.queue_capacity:
            hospital.queue.append(pid)
            patient.in_queue = True
            wait_limit = safe_waiting_time(patient.priority_score)
            schedule(t + wait_limit, "QUEUE_TIMEOUT", pid)
        else:
            reject(patient, "queue_full", t)

    def handle_queue_timeout(t, pid):
        patient = hospital.patients[pid]
        if not patient.in_queue:
            return  # stale event: patient already admitted or already removed
        hospital.queue.remove(pid)
        reject(patient, "timeout", t)

    def handle_discharge(t, pid):
        patient = hospital.patients[pid]
        patient.discharge_time = t
        patient.total_time_in_system = t - patient.arrival_time
        hospital.free_bed_numbers.append(patient.bed_number)
        hospital.free_bed_numbers.sort()
        hospital.beds_used -= 1
        if hospital.queue:                      # FIFO baseline discipline
            next_pid = hospital.queue.pop(0)
            admit(hospital.patients[next_pid], t)

    def handle_monitor(t):
        beds_used = hospital.beds_used
        queue_length = len(hospital.queue)
        hospital.hourly_records.append({
            "time": t,
            "beds_used": beds_used,
            "available_beds": hospital.num_beds - beds_used,
            "queue_length": queue_length,
            "bed_utilization": beds_used / hospital.num_beds,
            "queue_full": queue_length >= hospital.queue_capacity,
            "system_pressure": beds_used + queue_length,
            "over_capacity_pressure": (beds_used + queue_length) > hospital.num_beds,
        })
        if t < horizon:
            schedule(t + 1, "MONITOR")

    schedule(0.0, "ARRIVAL")
    schedule(0.0, "MONITOR")

    handlers = {
        "ARRIVAL": lambda t, pid: handle_arrival(t),
        "TRIAGE_DONE": handle_triage_done,
        "QUEUE_TIMEOUT": handle_queue_timeout,
        "DISCHARGE": handle_discharge,
        "MONITOR": lambda t, pid: handle_monitor(t),
    }

    # ---- main DES loop: pop nearest event, advance clock, dispatch ----
    while fel:
        t, _, event_type, pid = heapq.heappop(fel)
        handlers[event_type](t, pid)

    patient_df = pd.DataFrame([asdict(p) for p in hospital.patients.values()]).drop(columns=["in_queue"])
    hourly_df = pd.DataFrame(hospital.hourly_records)
    return patient_df, hourly_df


## 6. Section 4 Metrics

In [ ]:
def compute_metrics(patient_df, hourly_df, num_beds=NUM_BEDS):
    total = len(patient_df)
    admitted = patient_df[patient_df.status == "admitted"]
    metrics = {}
    metrics["admission_rate"] = len(admitted) / total if total else np.nan
    metrics["rejection_rate"] = (patient_df.status == "rejected").sum() / total if total else np.nan
    metrics["avg_waiting_time"] = admitted["wait_to_bed"].mean()
    metrics["avg_bed_utilization"] = hourly_df["bed_utilization"].mean()
    metrics["peak_queue_length"] = hourly_df["queue_length"].max()
    metrics["pct_time_queue_full"] = hourly_df["queue_full"].mean()
    metrics["pct_time_under_pressure"] = hourly_df["over_capacity_pressure"].mean()
    metrics["avg_length_of_stay"] = (admitted["discharge_time"] - admitted["arrival_time"]).mean()
    for level in ["low", "medium", "high"]:
        sub = patient_df[patient_df.priority_level == level]
        metrics[f"rejection_rate_{level}"] = (sub.status == "rejected").mean() if len(sub) else np.nan
    return metrics


## 7. Verification Checks (Appendix B)

Run automatically after every replication; a failed check raises immediately rather than
silently producing bad output.

In [ ]:
def verify_replication(patient_df, hourly_df, num_beds=NUM_BEDS, queue_capacity=QUEUE_CAPACITY):
    issues = []
    if (hourly_df["beds_used"] > num_beds).any():
        issues.append("Beds used exceeded capacity at some point.")
    if (hourly_df["queue_length"] > queue_capacity).any():
        issues.append("Queue length exceeded capacity at some point.")
    admitted = patient_df[patient_df.status == "admitted"]
    if admitted[["bed_number", "admission_time", "discharge_time"]].isnull().any().any():
        issues.append("Some admitted patients missing bed_number/admission_time/discharge_time.")
    rejected = patient_df[patient_df.status == "rejected"]
    if rejected["rejection_reason"].isnull().any():
        issues.append("Some rejected patients missing rejection_reason.")
    for bed_num, group in admitted.groupby("bed_number"):
        intervals = sorted(zip(group["admission_time"], group["discharge_time"]))
        for i in range(1, len(intervals)):
            if intervals[i][0] < intervals[i - 1][1]:
                issues.append(f"Bed {bed_num} double-booked.")
                break
    return issues


## 8. One Representative Base Run

Used for the patient-level / hourly-monitoring output tables and the required
time-series-style plots (Section 7).

In [ ]:
patient_df, hourly_df = run_single_replication(seed=BASE_SEED)

issues = verify_replication(patient_df, hourly_df)
assert not issues, f"Verification failed: {issues}"
print(f"Representative run: {len(patient_df)} patients simulated, 0 verification issues.")

patient_df.head()


In [ ]:
hourly_df.head()


In [ ]:
representative_metrics = compute_metrics(patient_df, hourly_df)
pd.Series(representative_metrics, name="value").to_frame()


## 9. Required Visualizations (base run)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly_df["time"], hourly_df["bed_utilization"], linewidth=0.8)
ax.set_xlabel("Time (hours)"); ax.set_ylabel("Bed utilization")
ax.set_title("Bed Utilization Over Time — Base Run")
fig.tight_layout()
fig.savefig("bed_utilization_over_time.png", dpi=150)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(hourly_df["time"], hourly_df["queue_length"], color="darkorange", linewidth=0.8)
ax.set_xlabel("Time (hours)"); ax.set_ylabel("Queue length")
ax.set_title("Queue Length Over Time — Base Run")
fig.tight_layout()
fig.savefig("queue_length_over_time.png", dpi=150)
plt.show()


In [ ]:
status_counts = patient_df["status"].value_counts()
# 'censored' will be 0 under the drain convention (Section 3.5) but included for completeness
for s in ["admitted", "rejected", "censored"]:
    if s not in status_counts:
        status_counts[s] = 0

fig, ax = plt.subplots(figsize=(6, 4))
status_counts.reindex(["admitted", "rejected", "censored"]).plot(kind="bar", ax=ax, color=["#4C72B0", "#C44E52", "#999999"])
ax.set_ylabel("Number of patients"); ax.set_title("Patient Outcomes — Base Run")
fig.tight_layout()
fig.savefig("patient_outcomes.png", dpi=150)
plt.show()


In [ ]:
rejection_counts = patient_df.loc[patient_df.status == "rejected", "rejection_reason"].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
if len(rejection_counts):
    rejection_counts.plot(kind="bar", ax=ax, color="#C44E52")
ax.set_ylabel("Count"); ax.set_title("Rejection Reasons — Base Run")
fig.tight_layout()
fig.savefig("rejection_reasons.png", dpi=150)
plt.show()


In [ ]:
admitted = patient_df[patient_df.status == "admitted"]

fig, ax = plt.subplots(figsize=(9, 5))
admitted.boxplot(column="treatment_duration", by="disease", ax=ax, rot=20)
ax.set_ylabel("Treatment duration (hours)")
ax.set_title("Treatment Duration by Disease — Base Run")
plt.suptitle("")
fig.tight_layout()
fig.savefig("treatment_duration_by_disease.png", dpi=150)
plt.show()


## 10. Statistical Analysis — 30+ Independent Replications

Each replication uses a distinct seed (`BASE_SEED*1000 + i`). Metrics are computed
per-replication, then aggregated: sample mean and 95% confidence interval **across
replications** (not across individual patients within one run).

In [ ]:
def run_replications(n_reps=N_REPLICATIONS, base_seed=BASE_SEED, **sim_kwargs):
    rows = []
    for i in range(n_reps):
        seed = base_seed * 1000 + i
        p_df, h_df = run_single_replication(seed=seed, **sim_kwargs)
        issues = verify_replication(p_df, h_df,
                                     num_beds=sim_kwargs.get("num_beds", NUM_BEDS),
                                     queue_capacity=sim_kwargs.get("queue_capacity", QUEUE_CAPACITY))
        if issues:
            raise AssertionError(f"Replication {i} (seed {seed}) failed verification: {issues}")
        metrics = compute_metrics(p_df, h_df, num_beds=sim_kwargs.get("num_beds", NUM_BEDS))
        metrics["replication"] = i
        metrics["seed"] = seed
        rows.append(metrics)
    return pd.DataFrame(rows)


def summarize_with_ci(replication_df, confidence=0.95):
    metric_cols = [c for c in replication_df.columns if c not in ("replication", "seed")]
    rows = []
    for col in metric_cols:
        vals = replication_df[col].dropna().values
        n = len(vals)
        mean = vals.mean()
        if n > 1:
            margin = stats.sem(vals) * stats.t.ppf((1 + confidence) / 2, n - 1)
        else:
            margin = np.nan
        rows.append({"metric": col, "mean": mean, "ci_lower": mean - margin,
                     "ci_upper": mean + margin, "n_replications": n})
    return pd.DataFrame(rows)


In [ ]:
base_replications = run_replications(n_reps=N_REPLICATIONS, base_seed=BASE_SEED)
base_replications.head()


In [ ]:
base_summary = summarize_with_ci(base_replications)
base_summary


## 11. Save Required Output Files

Per Section 8's "Output files" deliverable: patient records, hourly records,
replication-level metrics. (Scenario summary file will be added once the alternative
scenarios are built.)

In [ ]:
patient_df.to_csv("patient_level_base_run.csv", index=False)
hourly_df.to_csv("hourly_monitoring_base_run.csv", index=False)
base_replications.to_csv("replication_metrics_base_case.csv", index=False)
base_summary.to_csv("summary_ci_base_case.csv", index=False)

print("Saved: patient_level_base_run.csv, hourly_monitoring_base_run.csv,")
print("       replication_metrics_base_case.csv, summary_ci_base_case.csv")


## Next Steps (not yet in this notebook)

- Capacity experiment (10 → 14 beds)
- Queue-policy experiment (priority queue instead of FIFO)
- Demand-change experiment (faster arrivals)
- Scenario comparison table + confidence-interval comparison plot across all 4 scenarios
